# VolleyVision: fine-tune TrackNetV3 on your clips

Starts from the volleyball TrackNet checkpoint and trains on 17 of your clips (3 more for validation).
**Never trained on:** your 10 labeled clips (test7, 10, 12, 17, 21, 29, 30, 34, 36, 42), so the benchmark stays fair.

**Before running**, put these in Google Drive under `MyDrive/VolleyVision/`:
- `tracknet_data.tar` (the training set, about 475 MB)
- `tracknet_best.pt` (the starting checkpoint, from `models/` on your Mac)
- `patch_tracknet.py`

Runtime → Change runtime type → **GPU** (T4 is enough; L4 or A100 run faster). Then Runtime → Run all.
Checkpoints save to `MyDrive/VolleyVision/tracknet_ft/` after every epoch, so a disconnect loses at most one epoch (see the last cell).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/VolleyVision'
import os
for f in ['tracknet_data.tar', 'tracknet_best.pt', 'patch_tracknet.py']:
    print(('OK      ' if os.path.exists(f'{D}/{f}') else 'MISSING ') + f)

In [ ]:
%cd /content
!git clone -q https://github.com/qaz812345/TrackNetV3.git && cd TrackNetV3 && git checkout -q 6eda442
!pip install -q parse pycocotools
# unpack the training set onto the fast local disk
!mkdir -p /content/data && tar -xf {D}/tracknet_data.tar -C /content/data
!python {D}/patch_tracknet.py /content/TrackNetV3 /content/data/tracknet_data
!cat /content/data/tracknet_data/manifest.json | head -30

## Train
15 epochs at learning rate 1e-4 (fine-tuning, so lower than training from scratch).
On a T4 this should take roughly 5–10 min per epoch; the first epoch also builds an index of the data.
`TrackNet_best.pt` is the epoch with the best validation accuracy.

In [ ]:
%cd /content/TrackNetV3
OUT = f'{D}/tracknet_ft'
os.environ['INIT_CKPT'] = f'{D}/tracknet_best.pt'
!python train.py --model_name TrackNet --seq_len 8 --bg_mode concat --alpha 0.5 \
    --epochs 15 --batch_size 10 --learning_rate 1e-4 --save_dir {OUT} --verbose

## Done
Copy `MyDrive/VolleyVision/tracknet_ft/TrackNet_best.pt` into `Volley-Vision/models/tracknet/` on your Mac, renamed to
`FT_TrackNet_best.pt`. Then tell Claude, which scores it against your 10 labeled clips.

## If the runtime disconnected
Re-run the setup cells (everything above **Train**), then run this cell. It continues from the last saved epoch.

In [ ]:
%cd /content/TrackNetV3
!python train.py --model_name TrackNet --epochs 15 --save_dir {D}/tracknet_ft --resume_training --verbose